# Corpus analysis

Position statistics for a Corpus of Annotations. Every number here comes from the `bjj_analysis` module, which is tested on synthetic Corpora; this notebook only calls it and plots.

**Selection bias.** The Corpus covers only filmed, published Matches. Whole Brackets are annotated to avoid hand-picking, but events that are not filmed or not published are missing entirely.

## Load the Corpus

Annotations are read from `annotations/` at the repo root. Each file is checked against the Annotation schema and the vocabulary; files that fail are listed below instead of being used. Re-labels (`relabel_of` set) are left out. Set the metadata filters to stratify, e.g. `FILTERS = {"ruleset": "adcc", "gi": False}`.

While `annotations/` is empty, the notebook falls back to the fictional sample Annotation in `vocab/examples/` so every cell still runs.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import bjj_analysis as bjj

REPO = Path.cwd().resolve().parent if Path.cwd().name == "analysis" else Path.cwd().resolve()
CORPUS_DIR = REPO / "annotations"
FILTERS = {}  # e.g. {"ruleset": "adcc", "gi": False, "weight_class": "-88kg"}

corpus = bjj.load_corpus(CORPUS_DIR, **FILTERS)
if corpus.matches.empty and corpus.rejected.empty:
    print(f"No Annotations in {CORPUS_DIR}; using the fictional sample instead.")
    corpus = bjj.load_corpus(REPO / "vocab" / "examples", pattern="sample-annotation.json", **FILTERS)

print(f"{len(corpus.matches)} Matches, {len(corpus.segments)} Segments, {len(corpus.rejected)} rejected files")

In [ ]:
corpus.rejected

In [ ]:
corpus.matches[["annotation_id", "event", "bracket", "round", "ruleset", "gi", "weight_class", "athlete_a", "athlete_b", "winner", "method"]]

## Next-Position distributions

For each Position, where earned Transitions lead next (first-order). Resets are path breaks and never count. `count` / `position_total` are the raw counts; `probability` is smoothed with a Dirichlet prior pooled by Family, so a Position seen only a few times borrows strength from its Family (`family_count` / `family_total`). Rows with `count` 0 are next Positions seen only from the Family.

In [ ]:
PRIOR_STRENGTH = 2.0         # α: pseudo-counts borrowed from the Family
FAMILY_PRIOR_STRENGTH = 1.0  # β: pseudo-counts spread evenly over every Position

next_positions = bjj.next_position_distribution(
    corpus, prior_strength=PRIOR_STRENGTH, family_prior_strength=FAMILY_PRIOR_STRENGTH
)
next_positions.round(3)

## Transition types into each next Position

In [ ]:
next_transitions = bjj.next_position_transitions(corpus)
next_transitions.round(3)

## Positional Eval

For each Position, the chance that the Athlete in it wins, loses or draws the Match, given only the Position: like a chess engine's evaluation bar. The perspective is the first Role holder (`role`) for asymmetric Positions and the Initiator for symmetric ones. Symmetric Segments with no Initiator (the Match start, after a Reset) and Matches with no Result don't count.

Each Segment weighs `min(duration, cap) / cap`, summed per Match × Position × Athlete and capped at 1, so a brief hold counts less than a sustained one and no Match counts more than once per Position per Athlete. The estimate is a Dirichlet-multinomial pooled from Position into Family: `p_*` is the posterior mean, `p_*_low` / `p_*_high` the 90% credible interval, and `family_p_*` the Family mean it is pulled towards. `matches`, `segments` and `weight` are the sample sizes behind each row.

In [ ]:
CAP = 30.0                  # seconds for a Segment to count fully
EVAL_PRIOR_STRENGTH = 2.0   # α: pseudo-counts borrowed from the Family
EVAL_FAMILY_STRENGTH = 1.0  # β: pseudo-counts spreading each Family's mean evenly over win / loss / draw
EVAL_BASE_STRENGTH = 1.0    # γ: pseudo-counts every Position keeps spread evenly
EVAL_OPTIONS = dict(
    prior_strength=EVAL_PRIOR_STRENGTH,
    family_prior_strength=EVAL_FAMILY_STRENGTH,
    base_prior_strength=EVAL_BASE_STRENGTH,
)

positional_eval = bjj.positional_eval(corpus, cap=CAP, **EVAL_OPTIONS)
positional_eval.drop(columns=["perspective"]).sort_values("p_win", ascending=False).round(3)

### Sensitivity to the cap

P(win) for the headline perspective at caps of 15s, 30s and 60s. If a Position's ranking changes between columns, the conclusion depends on how brief holds are weighted.

In [ ]:
sensitivity = bjj.eval_sensitivity(corpus, caps=(15.0, 30.0, 60.0), **EVAL_OPTIONS)
sensitivity.pivot(index="position_id", columns="cap", values=["p_win", "weight"]).round(3)

## Retention

How long each Position is held, and whether the holder leaves it for something better, worse or neutral. The holder is the Athlete the Position's headline Positional Eval is about: the first Role holder (`role`), or the Initiator of a symmetric Position.

Each earned exit compares the holder's Positional Eval in the Position left with their Eval in the Position entered, from whichever side they hold there. It is `better` if the new 90% credible interval for P(win) lies wholly above the old one, `worse` if wholly below, and `neutral` if they overlap. With a small Corpus most intervals are wide, so most exits read as neutral. `unjudged` exits have no holder (a symmetric Position entered at the Match start or by a Reset). Resets and the Match end are counted apart (`resets`, `match_ends`): they are not exits the holder chose. `median_hold` and `total_hold` are in seconds.

In [ ]:
retention = bjj.retention(corpus, cap=CAP, **EVAL_OPTIONS)
retention.sort_values("median_hold", ascending=False).round(3)

### Exits, one by one

Every earned exit with the holder's P(win) interval before and after, and the judgement. Filter by `position_id` to see why a Position's exits were judged the way they were.

In [ ]:
exits = bjj.retention_exits(corpus, cap=CAP, **EVAL_OPTIONS)
exits[exits["exit"] == "earned"].round(3)

## Payoff

The Payoff rate: P(any Score Event or submission finish, for either Athlete, within 30s of entering a Position, or before the next Reset if that comes first). The window runs on past the Segment's end: this is where entering a Position leads, not what happens while it's held. A Match won by submission counts as a finish at its end even if no submission-finish Score Event was recorded.

`entries` / `entries_with_payoff` / `observed_share` are the raw counts. The estimate treats each Match as one cluster (weight 1 per Position, shared among its entries; `weight` is the total) and is pulled towards the Family's share (`family_p_payoff`). `p_payoff_low` / `p_payoff_high` bound the 90% credible interval. A Position entered in the last 30s of a Match has a shorter window, which pulls its rate down.

In [ ]:
HORIZON = 30.0  # seconds after entering a Position

payoff = bjj.payoff_rates(corpus, horizon=HORIZON)
payoff.sort_values("p_payoff", ascending=False).round(3)

## Sanity check against Lamas et al. 2024

Lamas et al., "No-gi Brazilian jiu-jitsu: A Markovian analysis of elite-level combat dynamics", *International Journal of Sports Science & Coaching* 19(4), 2024 ([doi:10.1177/17479541231210979](https://doi.org/10.1177/17479541231210979)), analysed all 93 Matches of the World Submission Fighting Championship 2019.

**Only the figures in the paper's abstract are used here.** The full text, with its per-action transition tables, is paywalled and has not been checked, so no Family-level tables from it are reproduced. The abstract gives two figures with a Family-level analogue in this Corpus:

- a back-take leads directly to a submission with probability 0.45 (the highest of any action);
- submission attempts have a point estimate of 1.03 per match, the only main action above one per competitor per match.

The abstract's other two figures, guard pass → guard pass (0.30) and takedown attempt → submission attempt (0.15), are about actions and Attempts, which this Corpus doesn't record, so they are left out.

Lamas et al. code actions, not Positions, and the event, year and ruleset differ, so treat this as a check for gross labelling errors (being off by a factor of several), not as a replication. With the fictional sample or a single Bracket the Corpus figures rest on very few Transitions: read `corpus_basis` before the value.

In [ ]:
bjj.lamas_2024_comparison(corpus).round(3)

## Network graph

Nodes are Positions (coloured by Family, sized by Segment count); edges are earned Transitions, wider for more frequent ones.

In [ ]:
graph = bjj.transition_network(corpus)
fig, ax = plt.subplots(figsize=(12, 9))
bjj.plot_transition_network(graph, ax=ax)
ax.set_title("Earned Transitions between Positions")
plt.show()

### Network coloured by Positional Eval

The same graph, with each node coloured by P(win) for its headline perspective (red = the Athlete in it tends to lose, blue = tends to win).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
bjj.plot_transition_network(graph, ax=ax, node_color=dict(zip(positional_eval["position_id"], positional_eval["p_win"])))
ax.set_title("Positions coloured by Positional Eval, P(win)")
plt.show()

## Self-consistency

Each blind re-label (`relabel_of` set) compared with its original Annotation. Re-labels are left out of every table above; they are loaded here only for this comparison.

- `pairs`: the share of Match time where both versions give the same Family, the same Position, and the same Position with the same Role holder, plus how well the Transition sequences align (1 − edit distance / longer sequence).
- `positions` and `transitions`: the Positions and Transition types with the most disagreement, first. Tighten their Label Guide entries.
- `confusions`: which Position the re-label gave instead.

The same report is available from the command line: `.venv/bin/bjj-agreement annotations/`.

In [ ]:
agreement = bjj.agreement_report(bjj.load_corpus(CORPUS_DIR, include_relabels=True, **FILTERS))
if agreement.pairs.empty:
    print("No re-label pairs in the Corpus yet.")
if not agreement.unpaired.empty:
    display(agreement.unpaired)
display(agreement.pairs.round(3))
display(agreement.positions.round(3))
display(agreement.transitions.round(3))
agreement.confusions.round(3)